[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 实操 1：更多的猫狗！


这次你将使用 [O. M. Parkhi 等人 2012](http://www.robots.ox.ac.uk/~vgg/publications/2012/parkhi12a/parkhi12a.pdf) 发布的 [Oxford-IIIT Pet Dataset](http://www.robots.ox.ac.uk/~vgg/data/pets/)，它包含 12 个猫品种和 25 个狗品种。你需要把第一课的代码改造成适合这个新任务，也就是 37 类分类。


##  导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
import torchvision
from torchvision import models,transforms,datasets
import time
%matplotlib inline

In [ ]:
torch.__version__

In [ ]:
import sys
sys.version

检查 GPU 是否可用，如果不可用就换一个 [runtime](https://jovianlin.io/pytorch-with-gpu-in-google-colab/)。


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print('Using gpu: %s ' % torch.cuda.is_available())

## 下载数据

[Oxford-IIIT Pet Dataset](http://www.robots.ox.ac.uk/~vgg/data/pets/) 网站上提供的数据由两个文件组成：`images.tar.gz` 和 `annotations.tar.gz`。我们首先需要下载并解压这些文件。

根据你用的是 google colab 还是自己的电脑，可以调整下面的代码来选择数据存放的位置。

要查看当前所在目录，可以用标准的 unix 命令：


In [ ]:
%pwd

如果想切换到一个目录来存放数据：


In [ ]:
%cd /home/mlelarge/
##路径

In [ ]:
%pwd

In [ ]:
#%mkdir data
# 如果不是在 google colab 上运行，下面这一行需要修改
%cd ./data/practical1/

## 警告

如果你在自己电脑上运行这个 notebook，只需要下载一次数据。如果还想再运行一次，可以放心跳过这一节和下面那一节，因为数据集已经好好地存在你电脑上了。

如果你在 google colab 上运行，每次运行都需要下载数据并做数据整理，因为一旦退出登录，数据就会被清空。


现在你已经在正确的目录里了，可以下载数据：


In [ ]:
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz

然后解压：


In [ ]:
!tar zxvf images.tar.gz
!tar zxvf annotations.tar.gz

检查一下是否一切正常！


In [ ]:
%ls

## 1. 练习：数据整理

为了使用 PyTorch 的 `dataloader`，你需要先做一点[数据整理](https://en.wikipedia.org/wiki/Data_wrangling)，把数据集组织好。

如果想了解文件是如何组织的，看一下 `annotations` 文件夹里的 `README` 文件。

首先，我们需要把数据集分成测试集和训练/验证集。为此，可以使用 `annotations/test.txt` 和 `annotations/trainval.txt` 这两个文件，它们包含原论文中测试集和训练/验证集里的图片文件名。


In [ ]:
!head annotations/test.txt

In [ ]:
!head annotations/trainval.txt

上面可以看到，原论文作者对数据集做了划分：`./images/Abyssinian_201.jpg` 在测试集里，而 `./images/Abyssinian_100.jpg` 在训练/验证集里，依此类推。

顺便说一句，如果你好奇 Abyssinian（阿比西尼亚猫）是什么，[这里](https://en.wikipedia.org/wiki/Abyssinian_cat)有解释。

我们先创建两个目录，分别存放来自 test 和 trainval 集合的图片。


In [ ]:
%mkdir test
%mkdir trainval

In [ ]:
%cd ..

In [ ]:
# 如果你想确认所有图片都在这里……
#%cd images/
#%ls

现在轮到你了！

所有图片都在 `./images/` 文件夹里，你需要按照下面的结构存放数据：
```bash
.
├── test
|   └── Abyssinian # 存放测试集中的 Abyssinian 图片
|   └── Bengal # 存放测试集中的 Bengal 图片
|    ... 
|   └── american_bulldog # 存放测试集中的 american bulldog 图片
|    ...
├── trainval
|   └── Abyssinian # 存放 trainval 集中的 Abyssinian 图片
|   └── Bengal # 存放 trainval 集中的 Bengal 图片
|    ...
|   └── american_bulldog # 存放 trainval 集中的 american bulldog 图片
|    ...
```

注意，名字以大写字母开头的都是猫，以小写字母开头的都是狗。

这里给出一种实现方法：逐行读取 `./annotations/test.txt` 文件；从每一行提取对应的文件名，然后把它从 `./images/filename_##.jpg` 复制到 `./test/filename/filename_##.jpg`，其中 `##` 是一个数字。

然后对 `trainval.txt` 文件做同样的操作。


下面是一小段代码，演示如何打开文件并逐行读取：


In [ ]:
with open('./annotations/test.txt') as fp:
    line = fp.readline()
    while line:
        f,_,_,_ = line.split(' ')
        print(f)
        line = fp.readline()
        break

要移除上面例子中的 `_201`，可以像下面这样用 `re` [正则表达式库](https://docs.python.org/3.6/library/re.html)：


In [ ]:
import re
pat = re.compile(r'_\d')
res,_ = pat.split(f)
print(res)

这个小片段可能有用：


In [ ]:
# 如果目录不存在则创建
def check_dir(dir_path):
    dir_path = dir_path.replace('//','/')
    os.makedirs(dir_path, exist_ok=True)

更多提示：
- 移动文件可以用 `shutil` 库，见[这里](https://docs.python.org/3.6/library/shutil.html#shutil.copy)
- 可以用 `os.path.join`
- 看看 python 的 [f-string](https://cito.github.io/blog/f-strings/)


In [ ]:
import shutil

In [ ]:
# 在这里写 test 的代码
path_test_dataset = 'test/'
with open('./annotations/test.txt') as fp:
    line = fp.readline()
    while line:
        f,_,_,_ = line.split(' ')
        res,_ = pat.split(f)
        path = os.path.join(path_test_dataset,res)
        check_dir(path)
        shutil.copy(f'./images/{f}.jpg',os.path.join(path,f'{f}.jpg'))
        line = fp.readline()

In [ ]:
# 在这里写 train 的代码
path_train_dataset = 'train/'
with open('./annotations/trainval.txt') as fp:
    line = fp.readline()
    while line:
        f,_,_,_ = line.split(' ')
        res,_ = pat.split(f)
        path = os.path.join(path_train_dataset,res)
        check_dir(path)
        shutil.copy(f'./images/{f}.jpg',os.path.join(path,f'{f}.jpg'))
        line = fp.readline()

## 数据处理


In [ ]:
%cd ..

现在你准备好重做第一课的内容了。

下面给出数据存放的路径。如果你在自己电脑上运行这段代码，需要修改这个单元格。


In [ ]:
data_dir = '/home/mlelarge/data/practical1'

`datasets` 是 `torchvision` 包中的一个类（见 [torchvision.datasets](http://pytorch.org/docs/master/torchvision/datasets.html)），负责数据加载。它内置了一个多线程加载器：从磁盘读取图片、按小批次（mini-batch）分组，并在网络的每次 _前向_/_反向_ 传播之后持续地把数据喂给 GPU。

图片在送入网络之前需要做一些预处理：尺寸必须统一为 $224\times 224 \times 3$，另外还要经过下面 normalize 变换所做的一些额外格式化（后面会解释）。


In [ ]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

vgg_format = transforms.Compose([
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                normalize,
            ])

In [ ]:
dsets = {x: datasets.ImageFolder(os.path.join(data_dir, x), vgg_format)
         for x in ['train', 'test']}

In [ ]:
os.path.join(data_dir,'train')

我们现在有 37 个不同的类别。


In [ ]:
dsets['train'].classes

In [ ]:
dsets['train'].class_to_idx

In [ ]:
dset_sizes = {x: len(dsets[x]) for x in ['train', 'test']}
dset_sizes

In [ ]:
dset_classes = dsets['train'].classes

`torchvision` 包支持对输入数据做复杂的预处理/变换（比如归一化、裁剪、翻转、抖动）。借助 `torchvision.transforms.Compose` 函数可以把一系列变换组合成一个流水线，见 [torchvision.transforms](http://pytorch.org/docs/master/torchvision/transforms.html)


In [ ]:
loader_train = torch.utils.data.DataLoader(dsets['train'], batch_size=64, shuffle=True, num_workers=6)#your code here

In [ ]:
loader_valid = torch.utils.data.DataLoader(dsets['test'], batch_size=5, shuffle=False, num_workers=6)#your code here

检查你的 dataloader，确认一切正常


In [ ]:
count = len(loader_valid)
inputs_try, labels_try = next(iter(loader_valid))

In [ ]:
count

In [ ]:
labels_try

In [ ]:
inputs_try.shape

一个显示图片的小函数：


In [ ]:
def imshow(inp, title=None):
#   用于显示 Tensor 的 imshow。
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = np.clip(std * inp + mean, 0,1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # 稍微暂停一下，让图像刷新出来

In [ ]:
# 从 batch 生成网格图像
out = torchvision.utils.make_grid(inputs_try)

imshow(out, title=[dset_classes[x] for x in labels_try])

In [ ]:
# 取一个训练数据 batch
inputs, classes = next(iter(loader_train))

n_images = 8

# 从 batch 生成网格图像
out = torchvision.utils.make_grid(inputs[0:n_images])

imshow(out, title=[dset_classes[x] for x in classes[0:n_images]])

## 2. 练习：修改 VGG 模型


torchvision 模块自带一个流行的 CNN 架构动物园，这些模型已经在 [ImageNet](http://www.image-net.org/)（120 万张训练图片）上训练过。第一次调用时，如果 `pretrained=True`，模型会从网上下载并保存到 `~/.torch/models`。
之后再调用会直接从本地读取。


In [ ]:
model_vgg = models.vgg16(weights='DEFAULT')#your code here

In [ ]:
inputs_try , labels_try = inputs_try.to(device), labels_try.to(device)

model_vgg = model_vgg.to(device)

In [ ]:
outputs_try = model_vgg(inputs_try)

In [ ]:
outputs_try

In [ ]:
outputs_try.shape

### 修改最后一层，并让所有层都不计算梯度


In [ ]:
print(model_vgg)

课程后面会学到这些不同的 block 各有什么作用。现在只需知道：

- 卷积层用于寻找图片中中小尺寸的模式——对图片做局部分析
- 全连接（Dense）层用于把整张图片上的模式组合起来——对图片做全局分析
- 池化层做下采样——减小图片尺寸，同时提高学到的特征的（平移）不变性


![vgg16](https://dataflowr.github.io/notebooks/Module1/img/vgg16.png)

这里我们的目标是使用已经训练好的模型，只改输出类别数。为此，把最后那个为 1000 类训练的 ```nn.Linear``` 层换成 37 类的。为了在训练时冻结其他层的权重，我们把 `requires_grad` 设为 `False`。这样反向传播时不会为这些层计算梯度，权重也就不会更新。只有 37 类的输出层权重会被更新。


PyTorch 关于 [LogSoftmax](https://pytorch.org/docs/stable/nn.html#logsoftmax) 的文档


In [ ]:
for param in model_vgg.parameters():
    param.requires_grad = False
# 在这里写你的代码
model_vgg.classifier._modules['6'] = nn.Linear(4096, 37)
model_vgg.classifier._modules['7'] = torch.nn.LogSoftmax(dim = 1)

In [ ]:
print(model_vgg.classifier)

一旦你修改了网络结构，别忘了把它放到设备（device）上！


In [ ]:
model_vgg = model_vgg.to(device)# your code here

## 训练全连接模块


### 创建损失函数和优化器

PyTorch 关于 [NLLLoss](https://pytorch.org/docs/stable/nn.html#nllloss) 和 [torch.optim 模块](https://pytorch.org/docs/stable/optim.html#module-torch.optim) 的文档


In [ ]:
criterion = nn.NLLLoss()
lr = 0.001
optimizer_vgg = torch.optim.SGD(model_vgg.classifier[6].parameters(),lr = lr)

### 训练模型


In [ ]:
def train_model(model,dataloader,size,epochs=1,optimizer=None):
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        running_corrects = 0
        for inputs,classes in dataloader:
            inputs = inputs.to(device)
            classes = classes.to(device)
            outputs = model(inputs)
            loss = criterion(outputs,classes)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            _,preds = torch.max(outputs.data,1)
            # 统计信息
            running_loss += loss.data.item()
            running_corrects += torch.sum(preds == classes.data)
        epoch_loss = running_loss / size
        epoch_acc = running_corrects.data.item() / size
        print('Loss: {:.4f} Acc: {:.4f}'.format(
                     epoch_loss, epoch_acc))

In [ ]:
%%time
train_model(model_vgg,loader_train,size=dset_sizes['train'],epochs=2,optimizer=optimizer_vgg)

In [ ]:
def test_model(model,dataloader,size):
    model.eval()
    predictions = np.zeros(size)
    all_classes = np.zeros(size)
    all_proba = np.zeros((size,37))
    i = 0
    running_loss = 0.0
    running_corrects = 0
    #print(size)
    for inputs,classes in dataloader:
        inputs = inputs.to(device)
        classes = classes.to(device)
        outputs = model(inputs)
        loss = criterion(outputs,classes)           
        _,preds = torch.max(outputs.data,1)
            # 统计信息
        running_loss += loss.data.item()
        running_corrects += torch.sum(preds == classes.data)
        predictions[i:i+len(classes)] = preds.to('cpu').numpy()
        all_classes[i:i+len(classes)] = classes.to('cpu').numpy()
        all_proba[i:i+len(classes),:] = outputs.data.to('cpu').numpy()
        i += len(classes)
    epoch_loss = running_loss / size
    epoch_acc = running_corrects.data.item() / size
    print('Loss: {:.4f} Acc: {:.4f}'.format(
                     epoch_loss, epoch_acc))
    return predictions, all_proba, all_classes

In [ ]:
predictions, all_proba, all_classes = test_model(model_vgg,loader_valid,size=dset_sizes['test'])

In [ ]:
# 取一个训练数据 batch
inputs, classes = next(iter(loader_valid))

out = torchvision.utils.make_grid(inputs[0:n_images])

imshow(out, title=[dset_classes[x] for x in classes[0:n_images]])

计算你的网络对 `inputs[:n_images]` 的预测及对应的概率。

提示：用 `torch.max` 和 `torch.exp`。

别忘了把输入放到设备上！


In [ ]:
outputs = model_vgg(inputs[:n_images].to(device))
print(torch.exp(outputs))

In [ ]:
# 在这里写你的代码
vals_try, preds_try = torch.max(outputs.data,1)# your code here

In [ ]:
preds_try

In [ ]:
classes[:n_images]

In [ ]:
torch.exp(vals_try)

## 通过预计算特征来加速训练


In [ ]:
def preconvfeat(dataloader):
    conv_features = []
    labels_list = []
    for data in dataloader:
        inputs,labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        x = model_vgg.features(inputs)
        conv_features.extend(x.data.cpu().numpy())
        labels_list.extend(labels.data.cpu().numpy())
    conv_features = np.concatenate([[feat] for feat in conv_features])
    return (conv_features,labels_list)

In [ ]:
%%time
conv_feat_train,labels_train = preconvfeat(loader_train)

In [ ]:
conv_feat_train.shape

In [ ]:
%%time
conv_feat_valid,labels_valid = preconvfeat(loader_valid)

### 创建新的数据生成器

我们不再加载图片了，所以需要自己构建数据加载器。


In [ ]:
dtype=torch.float
datasetfeat_train = [[torch.from_numpy(f).type(dtype),torch.tensor(l).type(torch.long)] for (f,l) in zip(conv_feat_train,labels_train)]
datasetfeat_train = [(inputs.reshape(-1), classes) for [inputs,classes] in datasetfeat_train]
loaderfeat_train = torch.utils.data.DataLoader(datasetfeat_train, batch_size=128, shuffle=True)

现在你可以训练更多个 epoch 了。


In [ ]:
%%time
train_model(model_vgg.classifier,dataloader=loaderfeat_train,size=dset_sizes['train'],epochs=80,optimizer=optimizer_vgg)

In [ ]:
datasetfeat_valid = [[torch.from_numpy(f).type(dtype),torch.tensor(l).type(torch.long)] for (f,l) in zip(conv_feat_valid,labels_valid)]
datasetfeat_valid = [(inputs.reshape(-1), classes) for [inputs,classes] in datasetfeat_valid]
loaderfeat_valid = torch.utils.data.DataLoader(datasetfeat_valid, batch_size=128, shuffle=False)

现在你可以计算测试集上的准确率了。


In [ ]:
predictions, all_proba, all_classes = test_model(model_vgg.classifier,dataloader=loaderfeat_valid,size=dset_sizes['test'])

## 混淆矩阵

对于 37 类分类，绘制混淆矩阵有助于查看算法在每个类别上的表现。


In [ ]:
#!pip install -U scikit-learn

In [ ]:
from sklearn.metrics import confusion_matrix
import itertools
def make_fig_cm(cm):
    fig = plt.figure(figsize=(12,12))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    tick_marks = np.arange(37);
    plt.xticks(tick_marks, dset_classes, rotation=90);
    plt.yticks(tick_marks, dset_classes, rotation=0);
    plt.tight_layout();
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        coeff = f'{cm[i, j]}'
        plt.text(j, i, coeff, horizontalalignment="center", verticalalignment="center", color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('Actual');
    plt.xlabel('Predicted');

In [ ]:
cm = confusion_matrix(all_classes,predictions)

In [ ]:
make_fig_cm(cm)

这里可以看到，[american pit bull terrier](https://en.wikipedia.org/wiki/American_Pit_Bull_Terrier) 经常被预测为 [staffordshire bull terrier](https://en.wikipedia.org/wiki/Staffordshire_Bull_Terrier)，但总体上你的算法应该会给出相当不错的结果！


## 3. 练习：更换神经网络模型

很好！到目前为止，你复现了第一课的结果。现在，你需要换一个模型。我建议从[这里](https://pytorch.org/vision/main/models.html)的列表里选一个 `resnet34`。

__注意__ 这个练习的主要目的是学习如何修改网络。我们并不真的在意性能……


In [ ]:
model_resnet = models.resnet34(weights='DEFAULT')# your code here

In [ ]:
print(model_resnet)

OK，结构和课程里看到的 VGG 很不一样，但我们仍然可以看到最后一层是一个 Linear 层：输入 512 维向量，输出 1000 维向量（也就是 Imagenet 的类别数）。因此你应该能把这个网络改造成你的 37 类分类任务！

首先冻结模型的权重，然后把最后一层替换成合适大小的 Linear 层。


In [ ]:
# 在这里写你的代码
#Hint 
print(model_resnet.fc)

In [ ]:
model_resnet.eval()
for param in model_resnet.parameters():
    param.requires_grad = False
# your code here
model_resnet.fc = nn.Linear(512, 37)

现在需要加上 LogSoftmax 层。按照[这里的说明](https://discuss.pytorch.org/t/how-to-add-an-additional-layer-on-top-of-a-pretrained-model/21303/2)添加这一层


In [ ]:
model_resnet_lsm = nn.Sequential(model_resnet, torch.nn.LogSoftmax(dim = 1))# your code here

检查一下是否一切正常！


In [ ]:
inputs_try , labels_try = inputs_try.to(device), labels_try.to(device)
model_resnet_lsm = model_resnet_lsm.to(device)
outputs_try = model_resnet_lsm(inputs_try)

In [ ]:
outputs_try.shape

现在可以开始训练了。

使用和之前一样的损失：[NLLLoss](https://pytorch.org/docs/stable/nn.html#nllloss)，但需要修改优化器的参数 [torch.optim 模块](https://pytorch.org/docs/stable/optim.html#module-torch.optim)


In [ ]:
#Hint print(model_resnet_lsm[0].fc)
print(model_resnet_lsm[0].fc)

In [ ]:
lr = 0.001
optimizer_resnet = torch.optim.SGD(model_resnet_lsm[0].fc.parameters(),lr = lr)#your code here

现在你可以用和上面一样的函数做训练循环。


In [ ]:
%%time
train_model(model_resnet_lsm,loader_train,size=dset_sizes['train'],epochs=30,optimizer=optimizer_resnet)

In [ ]:
model_resnet_lsm.eval()

In [ ]:
%%time
predictions, all_proba, all_classes = test_model(model_resnet_lsm,loader_valid,size=dset_sizes['test'])

In [ ]:
cm = confusion_matrix(all_classes,predictions)
make_fig_cm(cm)

## 干得漂亮！

最后一部分，你会怎么做来加速整个过程？


[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)